## BPE from schratch


https://sebastianraschka.com/blog/2025/bpe-from-scratch.html

In [26]:
from collections import Counter, deque # Para contar y manejar pares de tokens
class BPETokenizerSimple:
    def __init__(self):
        # VOCABULARIO PRINCIPAL (ID → Token)
        # Mapea cada ID numérico a su representación de texto
        # Ejemplo: {0: 'a', 1: 'b', 2: 'c', 257: 'th', 258: 'ing'}
        self.vocab = {}
        
        # VOCABULARIO INVERSO (Token → ID)  
        # Mapea cada token de texto a su ID numérico único
        # Ejemplo: {'a': 0, 'b': 1, 'c': 2, 'th': 257, 'ing': 258}
        self.inverse_vocab = {}
        
        # REGLAS DE FUSIÓN BPE
        # Diccionario que guarda qué pares de tokens se fusionaron
        # Clave: tupla (token_id1, token_id2) - par original
        # Valor: nuevo_token_id - resultado de la fusión
        # Ejemplo: {(116, 104): 257} significa que 't' + 'h' se convierte en 'th'
        self.bpe_merges = {}


## FUNCIÓN 1: ENCUENTRA EL PAR MÁS FRECUENTE
    

In [27]:
@staticmethod
def find_freq_pair(token_ids, mode="most"):
    # PASO 1: CREAR TODOS LOS PARES ADYACENTES
    # zip(token_ids, token_ids[1:]) crea pares de tokens consecutivos
    # Ejemplo: [1,2,3,4] → [(1,2), (2,3), (3,4)]
    pairs = Counter(zip(token_ids, token_ids[1:]))
    
    # PASO 2: VERIFICAR SI HAY PARES DISPONIBLES  
    if not pairs:
        # Si la lista está vacía o tiene un solo elemento, no hay pares
        return None
    
    # PASO 3: ENCONTRAR EL PAR SEGÚN EL CRITERIO
    if mode == "most":
        # max() encuentra el par con mayor frecuencia
        # key=lambda x: x[1] usa el conteo (segundo elemento) para comparar
        # [0] toma solo la tupla del par, no el conteo
        return max(pairs.items(), key=lambda x: x[1])[0]
    elif mode == "least":
        # min() encuentra el par con menor frecuencia (rara vez usado)
        return min(pairs.items(), key=lambda x: x[1])[0]
    else:
        raise ValueError("Modo inválido. Usa 'most' o 'least'.")
BPETokenizerSimple.find_freq_pair = find_freq_pair


## FUNCIÓN 2: REEMPLAZA TODAS LAS OCURRENCIAS DE UN PAR


In [28]:
@staticmethod 
def replace_pair(token_ids, pair_id, new_id):   
    # PASO 1: USAR DEQUE PARA PROCESAMIENTO EFICIENTE
    # deque permite agregar/quitar elementos del inicio/final eficientemente
    dq = deque(token_ids)
    replaced = []  # Lista resultado
    
    # PASO 2: PROCESAR CADA TOKEN SECUENCIALMENTE
    while dq:  # Mientras queden tokens por procesar
        # Tomar el primer token
        current = dq.popleft()
        
        # PASO 3: VERIFICAR SI FORMA EL PAR A REEMPLAZAR
        if dq and (current, dq[0]) == pair_id:
            # ¡ENCONTRAMOS EL PAR!
            # - dq[0] es el siguiente token 
            # - (current, dq[0]) forma exactamente el par que buscamos
            
            # Agregar el nuevo token fusionado
            replaced.append(new_id)
            
            # Quitar el segundo token del par (ya fue fusionado)
            dq.popleft() 
            
        else:
            # NO es el par que buscamos, mantener el token original
            replaced.append(current)
    
    return replaced
BPETokenizerSimple.replace_pair = replace_pair

## Método de Entrenamiento

In [29]:
def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
    

    # Preprocesamiento: Reemplaza espacios con "Ġ"
    # Nota que Ġ es una particularidad de la implementación BPE de GPT-2
    # Ej: "Hola mundo" puede tokenizarse como ["Hola", "Ġmundo"]
    processed_text = []
    for i, char in enumerate(text):
        if char == " " and i != 0:
            processed_text.append("Ġ")
        if char != " ":
            processed_text.append(char)
    processed_text = "".join(processed_text)

    # Inicializa vocabulario con caracteres únicos, incluyendo "Ġ" si está presente
    # Comienza con los primeros 256 caracteres ASCII
    unique_chars = [chr(i) for i in range(256)]
    unique_chars.extend(
        char for char in sorted(set(processed_text))
        if char not in unique_chars
    )
    if "Ġ" not in unique_chars:
        unique_chars.append("Ġ")

    self.vocab = {i: char for i, char in enumerate(unique_chars)}
    self.inverse_vocab = {char: i for i, char in self.vocab.items()}

    # Agrega tokens especiales permitidos
    if allowed_special:
        for token in allowed_special:
            if token not in self.inverse_vocab:
                new_id = len(self.vocab)
                self.vocab[new_id] = token
                self.inverse_vocab[token] = new_id

    # Tokeniza el texto procesado en IDs de tokens
    token_ids = [self.inverse_vocab[char] for char in processed_text]

    # Pasos 1-3 de BPE: Repetidamente encuentra y reemplaza pares frecuentes
    for new_id in range(len(self.vocab), vocab_size):
        pair_id = self.find_freq_pair(token_ids, mode="most")
        if pair_id is None:
            break
        token_ids = self.replace_pair(token_ids, pair_id, new_id)
        self.bpe_merges[pair_id] = new_id

    # Construye el vocabulario con tokens fusionados
    for (p0, p1), new_id in self.bpe_merges.items():
        merged_token = self.vocab[p0] + self.vocab[p1]
        self.vocab[new_id] = merged_token
        self.inverse_vocab[merged_token] = new_id

BPETokenizerSimple.train = train

## FUNCIÓN DE TOKENIZACIÓN BPE PARA TOKENS INDIVIDUALES

In [30]:
def tokenize_with_bpe(self, token):
    
    # ================================================================
    # PASO 1: CONVERSIÓN A CARACTERES INDIVIDUALES
    # ================================================================
    
    # Convierte cada carácter del token a su ID numérico
    # Ejemplo: "hola" → ['h', 'o', 'l', 'a'] → [104, 111, 108, 97]
    token_ids = [self.inverse_vocab.get(char, None) for char in token]
    
    # VERIFICACIÓN DE SEGURIDAD: ¿Todos los caracteres están en vocabulario?
    if None in token_ids:
        # Encuentra qué caracteres faltan para dar error informativo
        missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
        raise ValueError(f"Caracteres no encontrados en vocabulario: {missing_chars}")

    # ================================================================
    # PASO 2: APLICACIÓN ITERATIVA DE REGLAS BPE
    # ================================================================
    
    # Bandera que controla si aún se pueden hacer fusiones
    can_merge = True
    
    # BUCLE PRINCIPAL: Continúa mientras se puedan hacer fusiones
    while can_merge and len(token_ids) > 1:
        can_merge = False  # Asume que no habrá fusiones esta iteración
        new_tokens = []    # Lista para almacenar tokens después de fusiones
        i = 0              # Índice para recorrer la lista de tokens
        
        # ============================================================
        # PASO 2.1: BUSCAR Y APLICAR FUSIONES EN ESTA ITERACIÓN
        # ============================================================
        
        # Recorre todos los pares adyacentes de tokens
        while i < len(token_ids) - 1:
            # Crea par con token actual y siguiente
            pair = (token_ids[i], token_ids[i + 1])
            
            # ¿Este par tiene una regla de fusión aprendida?
            if pair in self.bpe_merges:
                # ¡SÍ! Aplicar la fusión
                
                # Obtiene el ID del token fusionado
                merged_token_id = self.bpe_merges[pair]
                
                # Agrega el token fusionado en lugar de los dos originales
                new_tokens.append(merged_token_id)
                
                # Salta ambos tokens del par (ya fueron fusionados)
                i += 2
                
                # Marca que hubo al menos una fusión en esta iteración
                can_merge = True
                
            else:
                # NO hay regla para este par, mantener token original
                new_tokens.append(token_ids[i])
                i += 1
        
        # ============================================================
        # PASO 2.2: MANEJAR TOKEN FINAL SI QUEDÓ SIN PROCESAR
        # ============================================================
        
        # Si el último token no fue parte de ningún par, agregarlo
        if i < len(token_ids):
            new_tokens.append(token_ids[i])
        
        # ============================================================
        # PASO 2.3: PREPARAR PARA SIGUIENTE ITERACIÓN
        # ============================================================
        
        # La nueva lista se convierte en la lista a procesar
        token_ids = new_tokens
        
        # El bucle continuará si can_merge=True (hubo fusiones)
        # y si quedan al menos 2 tokens para formar pares
    
    # ================================================================
    # PASO 3: RETORNAR RESULTADO FINAL
    # ================================================================
    
    # Retorna la lista de IDs después de aplicar todas las fusiones posibles
    return token_ids

# Asigna la función a la clase
BPETokenizerSimple.tokenize_with_bpe = tokenize_with_bpe


In [31]:
def encode(self, text, allowed_special=None):
    import re  # Importa regex para manejo de tokens especiales
    
    token_ids = []  # Lista final de IDs que se retornará
    
    # ===============================================================
    # FASE 1: MANEJO DE TOKENS ESPECIALES (SI ESTÁ HABILITADO)
    # ===============================================================
    if allowed_special is not None and len(allowed_special) > 0:
        
        # Construye patrón regex para encontrar tokens especiales
        # Los ordena por longitud (más largo primero) para evitar matches parciales
        # Ejemplo: si tienes "<|end|>" y "<|endoftext|>", busca primero el más largo
        special_pattern = (
            "(" + "|".join(re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)) + ")"
        )
        
        last_index = 0  # Rastrea posición en el texto
        
        # Busca cada ocurrencia de tokens especiales en el texto
        for match in re.finditer(special_pattern, text):
            # PROCESA TEXTO ANTES DEL TOKEN ESPECIAL
            prefix = text[last_index:match.start()]
            # Codifica recursivamente el prefijo SIN manejo de tokens especiales
            token_ids.extend(self.encode(prefix, allowed_special=None))
            
            # PROCESA EL TOKEN ESPECIAL ENCONTRADO
            special_token = match.group(0)
            if special_token in self.inverse_vocab:
                # Agrega directamente el ID del token especial
                token_ids.append(self.inverse_vocab[special_token])
            else:
                # Error si el token especial no está en vocabulario
                raise ValueError(f"Token especial {special_token} no encontrado en vocabulario.")
            
            # Actualiza posición para continuar después del token especial
            last_index = match.end()
        
        # Obtiene texto restante después del último token especial
        text = text[last_index:]
        
        # VERIFICACIÓN DE SEGURIDAD: Busca tokens especiales no permitidos
        # Encuentra tokens con formato <|...| > que no están en allowed_special
        disallowed = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
        ]
        if disallowed:
            raise ValueError(f"Tokens especiales no permitidos encontrados en texto: {disallowed}")
    
    # ===============================================================
    # FASE 2: TOKENIZACIÓN PRINCIPAL DEL TEXTO
    # ===============================================================
    
    tokens = []  # Lista temporal de tokens de texto
    
    # MANEJO DE MÚLTIPLES LÍNEAS
    lines = text.split("\n")  # Divide texto en líneas 
    
    for i, line in enumerate(lines):
        # Agrega símbolo de nueva línea para líneas después de la primera
        if i > 0:
            tokens.append("\n")
        
        # MANEJO DE PALABRAS DENTRO DE CADA LÍNEA
        words = line.split()  # Divide línea en palabras (separadas por espacios)
        
        for j, word in enumerate(words):
            # LÓGICA DE ESPACIOS CON SÍMBOLO "Ġ":
            # - Primera palabra después de nueva línea: lleva "Ġ" (representa espacio)
            # - Primera palabra de primera línea: sin "Ġ"
            # - Resto de palabras: llevan "Ġ" (representan espacios anteriores)
            
            if j == 0 and i > 0:
                # Primera palabra después de nueva línea
                tokens.append("Ġ" + word)
            elif j == 0:
                # Primera palabra de la primera línea
                tokens.append(word)
            else:
                # Palabras subsecuentes (llevan espacio representado por "Ġ")
                tokens.append("Ġ" + word)
    
    # ===============================================================
    # FASE 3: CONVERSIÓN DE TOKENS A IDs
    # ===============================================================
    
    for token in tokens:
        if token in self.inverse_vocab:
            # Token conocido: busca directamente su ID
            token_ids.append(self.inverse_vocab[token])
        else:
            # Token desconocido: aplica algoritmo BPE para descomponerlo
            # Esta llamada usa las reglas de fusión aprendidas durante entrenamiento
            token_ids.extend(self.tokenize_with_bpe(token))
    
    return token_ids  # Retorna lista final de IDs de tokens

BPETokenizerSimple.encode = encode



In [32]:
def decode(self, token_ids):
    
    # String que acumulará el texto final decodificado
    decoded_string = ""
    
    # ================================================================
    # BUCLE PRINCIPAL: PROCESA CADA TOKEN ID
    # ================================================================
    
    # Itera sobre cada ID de token con su índice
    for i, token_id in enumerate(token_ids):
        
        # ============================================================
        # PASO 1: VERIFICACIÓN DE SEGURIDAD
        # ============================================================
        
        # ¿Este ID existe en nuestro vocabulario?
        if token_id not in self.vocab:
            raise ValueError(f"Token ID {token_id} no encontrado en vocabulario.")
        
        # Obtiene el string correspondiente al ID
        token = self.vocab[token_id]  # Ej: 123 → "Hola"
        
        # ============================================================
        # PASO 2: MANEJO ESPECIAL DE SALTOS DE LÍNEA
        # ============================================================
        
        if token == "\n":
            # LÓGICA ESPECIAL PARA NUEVA LÍNEA:
            # Si no hay espacio antes del salto de línea, agregarlo
            if decoded_string and not decoded_string.endswith(" "):
                decoded_string += " "  # Agrega espacio para mejor formato
            
            # Agrega el salto de línea
            decoded_string += token
            
        # ============================================================
        # PASO 3: MANEJO DEL SÍMBOLO ESPECIAL "Ġ" (ESPACIOS)
        # ============================================================
        
        elif token.startswith("Ġ"):
            # TOKENS QUE REPRESENTAN ESPACIOS:
            # "Ġ" es el símbolo especial que representa espacios iniciales
            # Ejemplo: "Ġmundo" significa " mundo" (con espacio al inicio)
            
            # Convierte "Ġ" de vuelta a espacio real y agrega el resto del token
            decoded_string += " " + token[1:]  # "Ġmundo" → " mundo"
            
        # ============================================================
        # PASO 4: TOKENS NORMALES (SIN ESPACIOS ESPECIALES)
        # ============================================================
        
        else:
            # TOKENS REGULARES:
            # No tienen símbolos especiales, se agregan directamente
            # Ejemplo: "Hola", "123", "!", etc.
            decoded_string += token
    
    # ================================================================
    # PASO 5: RETORNAR RESULTADO FINAL
    # ================================================================
    
    # Retorna el string completamente reconstruido
    return decoded_string

# Asigna la función a la clase
BPETokenizerSimple.decode = decode


## Corpus Colombiano

In [33]:
import requests
import re
import time

def crear_corpus_colombia():
    """Descarga solo de Project Gutenberg y limpia espacios"""
    
    print("🇨🇴 CREANDO CORPUS COLOMBIA (SOLO GUTENBERG)...")
    corpus_total = ""
    
    # SOLO Project Gutenberg
    print("\n📚 Descargando de Project Gutenberg...")
    libros_es = [
        ("Cien años de soledad", "https://www.gutenberg.org/files/61733/61733-0.txt"), 
        ("Poesías (Bécquer)", "https://www.gutenberg.org/files/12847/12847-0.txt"),
        ("Platero y yo", "https://www.gutenberg.org/files/17946/17946-0.txt"),
    ]
    
    for i, (titulo, url) in enumerate(libros_es):
        try:
            print(f"   📖 {i+1}/{len(libros_es)}: {titulo}...")
            response = requests.get(url, timeout=15)
            response.encoding = 'utf-8'
            
            if response.status_code == 200:
                # LIMPIAR ESPACIOS DE GUTENBERG
                texto = response.text
                # Eliminar espacios múltiples y reemplazar con uno solo
                texto_limpio = re.sub(r'\s+', ' ', texto)  
                # Eliminar espacios al inicio y final de líneas
                texto_limpio = re.sub(r'^ +| +$', '', texto_limpio, flags=re.MULTILINE)
                
                corpus_total += f"\n=== {titulo} ===\n{texto_limpio}\n"
                print(f"   ✅ {len(texto_limpio):,} caracteres (limpio)")
            else:
                print(f"   ❌ Error: {response.status_code}")
            
            time.sleep(1)  # Pausa para no sobrecargar
            
        except Exception as e:
            print(f"   ❌ Error con {titulo}: {e}")
    
    # LIMPIEZA FINAL
    print("\n🧹 Limpieza final...")
    
    # Eliminar múltiples saltos de línea
    corpus_total = re.sub(r'\n{3,}', '\n\n', corpus_total)
    
    # Eliminar espacios múltiples una vez más
    corpus_total = re.sub(r'[ \t]+', ' ', corpus_total)
    
    # Eliminar líneas vacías excesivas
    corpus_total = re.sub(r'\n[ \t]*\n', '\n', corpus_total)
    
    # GUARDAR ARCHIVO
    print("💾 Guardando corpus...")
    
    with open('corpus_colombia.txt', 'w', encoding='utf-8') as f:
        f.write(corpus_total.strip())
    
    # ESTADÍSTICAS FINALES
    caracteres = len(corpus_total)
    palabras = len(corpus_total.split())
    
    print(f"\n✅ CORPUS COLOMBIA CREADO:")
    print(f"   📁 Archivo: corpus_colombia.txt")
    print(f"   📊 Caracteres: {caracteres:,}")
    print(f"   📰 Palabras: {palabras:,}")
    
    return corpus_total

# Ejecutar
corpus = crear_corpus_colombia()

🇨🇴 CREANDO CORPUS COLOMBIA (SOLO GUTENBERG)...

📚 Descargando de Project Gutenberg...
   📖 1/3: Cien años de soledad...
   ✅ 310,970 caracteres (limpio)
   📖 2/3: Poesías (Bécquer)...
   ✅ 502,430 caracteres (limpio)
   📖 3/3: Platero y yo...
   ✅ 77,022 caracteres (limpio)

🧹 Limpieza final...
💾 Guardando corpus...

✅ CORPUS COLOMBIA CREADO:
   📁 Archivo: corpus_colombia.txt
   📊 Caracteres: 890,502
   📰 Palabras: 167,036


In [34]:
# Cargar el corpus colombiano desde archivo
with open('corpus_colombia.txt', 'r', encoding='utf-8') as f:
    corpus_colombiano = f.read()

print(f"   Caracteres: {len(corpus_colombiano):,}")
print(f"   Palabras aproximadas: {len(corpus_colombiano.split()):,}")

   Caracteres: 890,500
   Palabras aproximadas: 167,036


In [35]:
tokenizer_colombiano = BPETokenizerSimple()
# Parámetros de entrenamiento
vocab_size =2000  # Tamaño del vocabulario final
tokens_especiales = {"<|endoftext|>"}  # Tokens especiales a incluir

print(f"\n⚙️ Configuración de entrenamiento:")
print(f"   Tamaño de vocabulario objetivo: {vocab_size}")
print(f"   Tokens especiales: {tokens_especiales}")



⚙️ Configuración de entrenamiento:
   Tamaño de vocabulario objetivo: 2000
   Tokens especiales: {'<|endoftext|>'}


In [36]:
import time

print(f"\n🏋️ Iniciando entrenamiento BPE...")

# Medir tiempo de entrenamiento
start_time = time.time()

# ¡ENTRENAR!
tokenizer_colombiano.train(
    text=corpus_colombiano,
    vocab_size=vocab_size,
    allowed_special=tokens_especiales
)

end_time = time.time()
training_time = end_time - start_time

print(f"\n✅ ¡Entrenamiento completado!")
print(f"   Tiempo transcurrido: {training_time:.2f} segundos")
print(f"   Vocabulario final: {len(tokenizer_colombiano.vocab)} tokens")
print(f"   Fusiones aprendidas: {len(tokenizer_colombiano.bpe_merges)} fusiones")



🏋️ Iniciando entrenamiento BPE...

✅ ¡Entrenamiento completado!
   Tiempo transcurrido: 127.76 segundos
   Vocabulario final: 2000 tokens
   Fusiones aprendidas: 1735 fusiones


In [37]:
# Ejemplos de prueba con texto colombiano
ejemplos_test = [
    "Gabriel García Márquez",
    "Bogotá Colombia",
    "café colombiano",
    "vallenato y bambuco",
    "Universidad Nacional"
]

print(f"\n🧪 Pruebas de tokenización:")
for i, ejemplo in enumerate(ejemplos_test, 1):
    # Codificar
    tokens = tokenizer_colombiano.encode(ejemplo)
    tokens_str = [tokenizer_colombiano.vocab[t] for t in tokens]
    
    # Decodificar para verificar
    decodificado = tokenizer_colombiano.decode(tokens)
    correcto = ejemplo == decodificado
    
    print(f"\n   Prueba {i}: '{ejemplo}'")
    print(f"   Tokens ({len(tokens)}): {tokens_str}")
    print(f"   IDs: {tokens}")
    print(f"   Decodificado: '{decodificado}'")
    print(f"   ✅ Correcto: {correcto}")



🧪 Pruebas de tokenización:

   Prueba 1: 'Gabriel García Márquez'
   Tokens (16): ['G', 'ab', 'ri', 'el', 'Ġ', 'G', 'ar', 'c', 'í', 'a', 'Ġ', 'M', 'á', 'r', 'qu', 'ez']
   IDs: [71, 419, 327, 309, 256, 71, 281, 99, 237, 97, 256, 77, 225, 114, 283, 592]
   Decodificado: 'Gabriel García Márquez'
   ✅ Correcto: True

   Prueba 2: 'Bogotá Colombia'
   Tokens (11): ['B', 'o', 'go', 't', 'á', 'Ġ', 'C', 'ol', 'om', 'bi', 'a']
   IDs: [66, 111, 485, 116, 225, 256, 67, 412, 306, 786, 97]
   Decodificado: 'Bogotá Colombia'
   ✅ Correcto: True

   Prueba 3: 'café colombiano'
   Tokens (9): ['c', 'af', 'é', 'Ġc', 'ol', 'om', 'bi', 'an', 'o']
   IDs: [99, 481, 233, 344, 412, 306, 786, 271, 111]
   Decodificado: 'café colombiano'
   ✅ Correcto: True

   Prueba 4: 'vallenato y bambuco'
   Tokens (13): ['v', 'al', 'le', 'n', 'at', 'o', 'Ġ', 'y', 'Ġb', 'am', 'bu', 'c', 'o']
   IDs: [118, 333, 463, 110, 364, 111, 256, 121, 516, 389, 1301, 99, 111]
   Decodificado: 'vallenato y bambuco'
   ✅ Correcto: T

In [38]:
# Prueba final con un texto más largo
texto_prueba = "asdaasdasdasdholadasdoafgoasdf"

print(f"\n🎯 Prueba final completa:")
print(f"Texto original ({len(texto_prueba)} chars):")
print(f"'{texto_prueba.strip()}'")

# Tokenizar
tokens_finales = tokenizer_colombiano.encode(texto_prueba)
tokens_str_finales = [tokenizer_colombiano.vocab[t] for t in tokens_finales]

print(f"\nTokenización ({len(tokens_finales)} tokens):")
print(f"Tokens: {tokens_str_finales}")

# Calcular ratio de compresión
ratio_compresion = len(texto_prueba) / len(tokens_finales)
print(f"\n📈 Ratio de compresión: {ratio_compresion:.2f} (chars/token)")

# Verificar reversibilidad
decodificado_final = tokenizer_colombiano.decode(tokens_finales)
print(f"\n🔄 Texto decodificado:")
print(f"'{decodificado_final.strip()}'")
print(f"✅ Reversibilidad correcta: {texto_prueba.strip() == decodificado_final.strip()}")




🎯 Prueba final completa:
Texto original (30 chars):
'asdaasdasdasdholadasdoafgoasdf'

Tokenización (18 tokens):
Tokens: ['as', 'd', 'a', 'as', 'd', 'as', 'd', 'as', 'd', 'ho', 'lad', 'as', 'do', 'af', 'go', 'as', 'd', 'f']

📈 Ratio de compresión: 1.67 (chars/token)

🔄 Texto decodificado:
'asdaasdasdasdholadasdoafgoasdf'
✅ Reversibilidad correcta: True


## Corpus de codigo

In [39]:
from datasets import load_dataset

print("📥 Descargando corpus Python...")
dataset = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)

corpus_codigo = ""
contador = 0

for ejemplo in dataset:
    # CONDICIÓN PARA PARAR: 1M caracteres O 1000 archivos
    if len(corpus_codigo) >= 1_000_000 or contador >= 1000:
        break
    
    codigo = ejemplo['content']
    if len(codigo) > 100:  # Solo código sustancial
        # Verificar que no exceda el límite al agregar
        if len(corpus_codigo) + len(codigo) <= 1_000_000:
            corpus_codigo += codigo + '\n\n'
            contador += 1
        else:
            # Si agregaría más de 1M, tomar solo la parte que cabe
            restante = 1_000_000 - len(corpus_codigo)
            if restante > 100:  # Solo si queda espacio suficiente
                corpus_codigo += codigo[:restante]
            break
    
    if contador % 100 == 0:
        print(f"📊 Procesados: {contador} archivos, Caracteres: {len(corpus_codigo):,}")

with open('corpus_codigo.txt', 'w', encoding='utf-8') as f:
    f.write(corpus_codigo)

print(f"✅ Corpus creado: {len(corpus_codigo):,} caracteres")

📥 Descargando corpus Python...
✅ Corpus creado: 1,000,000 caracteres


In [40]:
# Cargar el corpus de código desde archivo
with open('corpus_codigo.txt', 'r', encoding='utf-8') as f:
    corpus_codigo = f.read()

# Verificar que se cargó correctamente
print("💻 Corpus de Código Cargado:")
print(f"   Caracteres: {len(corpus_codigo):,}")
print(f"   Líneas de código: {len(corpus_codigo.splitlines()):,}")

💻 Corpus de Código Cargado:
   Caracteres: 1,000,000
   Líneas de código: 27,345


In [41]:
tokenizer_codigo = BPETokenizerSimple()
# Parámetros de entrenamiento para código
vocab_size_codigo = 2000  # Mismo tamaño que el tokenizador de texto para comparar
tokens_especiales_codigo = {"<|endoftext|>"}  # Tokens especiales

print(f"\n⚙️ Configuración de entrenamiento para código:")
print(f"   Tamaño de vocabulario objetivo: {vocab_size_codigo}")
print(f"   Tokens especiales: {tokens_especiales_codigo}")
print(f"   Fusiones a aprender: {vocab_size_codigo - 256 - len(tokens_especiales_codigo) - 1}")
print(f"   Diferencia con tokenizador texto: Especializándose en sintaxis de programación")


⚙️ Configuración de entrenamiento para código:
   Tamaño de vocabulario objetivo: 2000
   Tokens especiales: {'<|endoftext|>'}
   Fusiones a aprender: 1742
   Diferencia con tokenizador texto: Especializándose en sintaxis de programación


In [42]:
import time

print(f"\n🏋️ Iniciando entrenamiento BPE para código Python...")

# Medir tiempo de entrenamiento
start_time = time.time()

# ¡ENTRENAR PARA CÓDIGO!
tokenizer_codigo.train(
    text=corpus_codigo,
    vocab_size=vocab_size_codigo,
    allowed_special=tokens_especiales_codigo
)

end_time = time.time()
training_time = end_time - start_time

print(f"\n✅ ¡Entrenamiento de código completado!")
print(f"   Tiempo transcurrido: {training_time:.2f} segundos")
print(f"   Vocabulario final: {len(tokenizer_codigo.vocab)} tokens")
print(f"   Fusiones aprendidas: {len(tokenizer_codigo.bpe_merges)} fusiones")



🏋️ Iniciando entrenamiento BPE para código Python...

✅ ¡Entrenamiento de código completado!
   Tiempo transcurrido: 127.14 segundos
   Vocabulario final: 2000 tokens
   Fusiones aprendidas: 1724 fusiones


In [43]:
# Ejemplos de prueba con código típico
ejemplos_codigo_test = [
    "def calculate_fibonacci(n):",
    "import tensorflow as tf",
    "from sklearn import model_selection",
]

print(f"\n🧪 Pruebas de tokenización:")
for i, ejemplo in enumerate(ejemplos_codigo_test, 1):
    # Codificar
    tokens = tokenizer_codigo.encode(ejemplo)
    tokens_str = [tokenizer_codigo.vocab[t] for t in tokens]
    
    # Decodificar para verificar
    decodificado = tokenizer_codigo.decode(tokens)
    correcto = ejemplo == decodificado
    
    print(f"\n   Prueba {i}: '{ejemplo}'")
    print(f"   Tokens ({len(tokens)}): {tokens_str}")
    print(f"   IDs: {tokens}")
    print(f"   Decodificado: '{decodificado}'")
    print(f"   ✅ Correcto: {correcto}")




🧪 Pruebas de tokenización:

   Prueba 1: 'def calculate_fibonacci(n):'
   Tokens (17): ['de', 'f', 'Ġc', 'al', 'c', 'ul', 'at', 'e', '_f', 'i', 'bo', 'n', 'ac', 'ci', '(', 'n', '):']
   IDs: [291, 102, 818, 307, 99, 364, 340, 101, 1318, 105, 1864, 110, 361, 1893, 40, 110, 449]
   Decodificado: 'def calculate_fibonacci(n):'
   ✅ Correcto: True

   Prueba 2: 'import tensorflow as tf'
   Tokens (11): ['import', 'Ġ', 'ten', 'so', 'r', 'f', 'low', 'Ġ', 'as', 'Ġ', 'tf']
   IDs: [1606, 256, 687, 796, 114, 102, 1111, 256, 404, 256, 1647]
   Decodificado: 'import tensorflow as tf'
   ✅ Correcto: True

   Prueba 3: 'from sklearn import model_selection'
   Tokens (14): ['from', 'Ġ', 's', 'k', 'lear', 'n', 'Ġ', 'import', 'Ġm', 'od', 'el', '_s', 'el', 'ection']
   IDs: [480, 256, 115, 107, 1616, 110, 256, 1606, 1332, 861, 369, 456, 369, 1064]
   Decodificado: 'from sklearn import model_selection'
   ✅ Correcto: True


In [44]:
codigo_complejo = """
def train_deep_neural_network(X_train, y_train, layers=[128, 64, 32], dropout=0.2):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(layers[0], activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(layers[1], activation='relu'),
        tf.keras.layers.Dense(layers[2], activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
"""

print("CÓDIGO ORIGINAL:")
print(codigo_complejo)

# Codificar
tokens_complejo = tokenizer_codigo.encode(codigo_complejo)

# Decodificar
decodificado_complejo = tokenizer_codigo.decode(tokens_complejo)

print("CÓDIGO DECODIFICADO:")
print(decodificado_complejo)

print("\n" + "="*60)
print(f"¿Son idénticos? {codigo_complejo == decodificado_complejo}")


CÓDIGO ORIGINAL:

def train_deep_neural_network(X_train, y_train, layers=[128, 64, 32], dropout=0.2):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(layers[0], activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(layers[1], activation='relu'),
        tf.keras.layers.Dense(layers[2], activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

CÓDIGO DECODIFICADO:

 def train_deep_neural_network(X_train, y_train, layers=[128, 64, 32], dropout=0.2): 
 model = tf.keras.Sequential([ 
 tf.keras.layers.Dense(layers[0], activation='relu', input_shape=(X_train.shape[1],)), 
 tf.keras.layers.Dropout(dropout), 
 tf.keras.layers.Dense(layers[1], activation='relu'), 
 tf.keras.layers.Dense(layers[2], activation='relu'), 
 tf.keras.layers.Dense(1, activation='sigmoid') 
 ]) 


¿Son idénticos? False


## Comparaciones de BPEs 

## tokenizer_colombiano vs GPT2

In [45]:
import tiktoken
# Para texto colombiano
texto_colombiano = """Gabriel García Márquez escribió 'Cien años de soledad' en Cartagena. 
Bogotá es la capital de Colombia y el vallenato es patrimonio de la humanidad."""

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
tokens_gpt2_col = gpt2_tokenizer.encode(texto_colombiano)
decodificado_gpt2_col = gpt2_tokenizer.decode(tokens_gpt2_col)

tokens_col = tokenizer_colombiano.encode(texto_colombiano)
decodificado_col = tokenizer_colombiano.decode(tokens_col)

print(f"\n📥 CODIFICACIÓN GPT-2:")
print(f"   Número de tokens: {len(tokens_gpt2_col)}")
print(f"   IDs de tokens: {tokens_gpt2_col}")
print(f"\n📤 DECODIFICACIÓN GPT-2:")
print(f"   Texto reconstruido: '{decodificado_gpt2_col}'")

print(f"\n📥 CODIFICACIÓN BPE COLOMBIANO:")
print(f"   Número de tokens: {len(tokens_col)}")
print(f"   IDs de tokens: {tokens_col}")
print(f"\n📤 DECODIFICACIÓN BPE COLOMBIANO:")
print(f"   Texto reconstruido: '{decodificado_col}'")



📥 CODIFICACIÓN GPT-2:
   Número de tokens: 55
   IDs de tokens: [46079, 11719, 16364, 29690, 337, 6557, 81, 22281, 3671, 822, 72, 10205, 705, 34, 2013, 257, 12654, 418, 390, 1540, 276, 324, 6, 551, 13690, 363, 8107, 13, 220, 198, 33, 519, 313, 6557, 1658, 8591, 3139, 390, 21291, 331, 1288, 410, 439, 268, 5549, 1658, 1458, 3036, 261, 952, 390, 8591, 1692, 32482, 13]

📤 DECODIFICACIÓN GPT-2:
   Texto reconstruido: 'Gabriel García Márquez escribió 'Cien años de soledad' en Cartagena. 
Bogotá es la capital de Colombia y el vallenato es patrimonio de la humanidad.'

📥 CODIFICACIÓN BPE COLOMBIANO:
   Número de tokens: 100
   IDs de tokens: [71, 419, 327, 309, 256, 71, 281, 99, 237, 97, 256, 77, 225, 114, 283, 592, 256, 301, 653, 105, 786, 243, 256, 39, 67, 105, 273, 256, 97, 241, 363, 330, 101, 295, 412, 381, 410, 39, 256, 273, 256, 67, 281, 116, 368, 273, 97, 46, 10, 256, 66, 111, 485, 116, 225, 256, 301, 313, 97, 344, 472, 340, 333, 330, 101, 256, 67, 412, 306, 786, 97, 256, 121, 256, 309

## tokenizer_codigo vs GPT2

In [46]:
codigo_complejo = """
def train_deep_neural_network(X_train, y_train, layers=[128, 64, 32], dropout=0.2):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(layers[0], activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(layers[1], activation='relu'),
        tf.keras.layers.Dense(layers[2], activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
"""
gpt2_tokenizer = tiktoken.get_encoding("gpt2")
tokens_gpt2 = gpt2_tokenizer.encode(codigo_complejo)
decodificado_gpt2 = gpt2_tokenizer.decode(tokens_gpt2)

tokens_cod = tokenizer_codigo.encode(codigo_complejo)
decodificado_cod = tokenizer_codigo.decode(tokens_cod)

print(f"\n📥 CODIFICACIÓN GPT-2:")
print(f"   Número de tokens: {len(tokens_gpt2)}")
print(f"   IDs de tokens: {tokens_gpt2}")
print(f"\n📤 DECODIFICACIÓN GPT-2:")
print(f"   Texto reconstruido: '{decodificado_gpt2}'")


print(f"\n📥 CODIFICACIÓN BPE CÓDIGO:")
print(f"   Número de tokens: {len(tokens_cod)}")
print(f"   IDs de tokens: {tokens_cod}")
print(f"\n📤 DECODIFICACIÓN BPE CÓDIGO:")
print(f"   Texto reconstruido: '{decodificado_cod}'")


📥 CODIFICACIÓN GPT-2:
   Número de tokens: 203
   IDs de tokens: [198, 4299, 4512, 62, 22089, 62, 710, 1523, 62, 27349, 7, 55, 62, 27432, 11, 331, 62, 27432, 11, 11685, 41888, 12762, 11, 5598, 11, 3933, 4357, 4268, 448, 28, 15, 13, 17, 2599, 198, 220, 220, 220, 2746, 796, 48700, 13, 6122, 292, 13, 44015, 1843, 26933, 198, 220, 220, 220, 220, 220, 220, 220, 48700, 13, 6122, 292, 13, 75, 6962, 13, 35, 1072, 7, 75, 6962, 58, 15, 4357, 14916, 11639, 260, 2290, 3256, 5128, 62, 43358, 16193, 55, 62, 27432, 13, 43358, 58, 16, 4357, 36911, 198, 220, 220, 220, 220, 220, 220, 220, 48700, 13, 6122, 292, 13, 75, 6962, 13, 26932, 448, 7, 14781, 448, 828, 198, 220, 220, 220, 220, 220, 220, 220, 48700, 13, 6122, 292, 13, 75, 6962, 13, 35, 1072, 7, 75, 6962, 58, 16, 4357, 14916, 11639, 260, 2290, 33809, 198, 220, 220, 220, 220, 220, 220, 220, 48700, 13, 6122, 292, 13, 75, 6962, 13, 35, 1072, 7, 75, 6962, 58, 17, 4357, 14916, 11639, 260, 2290, 33809, 198, 220, 220, 220, 220, 220, 220, 220, 48700, 13, 